In [1]:
import os
import sys

# add the source directory to system path, so that relative imports work
# this fix is only for Jupyter Notebooks
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.integrate import solve_ivp
from scipy.optimize import least_squares

from orbital_mechanics.solar_system import SolarSystem
from orbital_mechanics.constants import ALTAIRA_MU, AU, YEAR

In [3]:
ss = SolarSystem()

# get id of PlanetX
planetX = ss.init_bodies['name'] == 'PlanetX'

# spacecraft initial state at t0
rv0 = np.zeros((6,))
rv0[0] = -200*AU

In [4]:
# dynamics function
def dydt(t,rv):
    r = rv[0:3]; v = rv[3:6]
    r_mag = np.linalg.norm(r)
    a = -ALTAIRA_MU/r_mag**3 * r
    dr = v; dv = a
    drv = np.concatenate([dr, dv])
    return drv

In [5]:
# distance from planetX
def rel_rv_from_X(t,rv):
    r = rv[0:3]; v = rv[3:6]

    # get planetX's position and velocity
    pl_rv = ss.get_state_at_t(t, planetX, 1e-12)
    pl_rv = pl_rv[['rx', 'ry', 'rz', 'vx', 'vy', 'vz']].to_numpy(dtype=np.float64).squeeze().ravel()

    pl_r = pl_rv[0:3]
    pl_v = pl_rv[3:6]

    rel_r = r - pl_r
    rel_v = v - pl_v

    return rel_r, rel_v

In [6]:
# event function (check periapsis distance to planetX)
def event(t,rv):
    rel_r, rel_v = rel_rv_from_X(t,rv)
    res = np.dot(rel_r, rel_v)

    return res

event.terminal = True

In [7]:
res = solve_ivp(dydt, [0, 200*YEAR], rv0, events=event, method='DOP853', rtol=1e-12, atol=1e-12)

In [8]:
vals = []
for i in range(len(res['t'])):
    rel_r, _ = rel_rv_from_X(res['t'][i],res['y'][:,i].ravel())
    rel_r_mag = np.linalg.norm(rel_r)
    dotp = event(res['t'][i], res['y'][:,i].ravel())

    vals.append([res['t'][i], rel_r_mag, dotp])

vals = np.array(vals, dtype=float)
print(vals)

[[ 0.00000000e+00  1.14079211e+10 -1.39928322e+10]
 [ 3.34665303e-01  1.14079211e+10 -1.39928322e+10]
 [ 3.68131833e+00  1.14079211e+10 -1.39928322e+10]
 [ 3.71478486e+01  1.14079211e+10 -1.39928320e+10]
 [ 3.71813152e+02  1.14079207e+10 -1.39928299e+10]
 [ 3.71846618e+03  1.14079166e+10 -1.39928094e+10]
 [ 3.71849965e+04  1.14078755e+10 -1.39926041e+10]
 [ 3.71850299e+05  1.14074650e+10 -1.39905507e+10]
 [ 3.71850333e+06  1.14033629e+10 -1.39700175e+10]
 [ 3.71850336e+07  1.13625921e+10 -1.37647210e+10]
 [ 3.71850337e+08  1.09809516e+10 -1.17168691e+10]
 [ 1.77604882e+09  9.97621083e+09 -3.36410917e+09]
 [ 2.37561215e+09  9.87539392e+09 -6.67572021e-06]]


In [9]:
# root func
def rootfunc(y0):
    yy0 = np.array([-200*AU, y0[0], y0[1], y0[2], 0.0, 0.0])
    res = solve_ivp(dydt, [0, 70*YEAR], yy0, events=event, method='DOP853', rtol=1e-12, atol=1e-12)
    rel_r, _ = rel_rv_from_X(res['t'][-1],res['y'][:,-1].ravel())
    print(res['t'][-1] / YEAR, rel_r)
    return rel_r

In [10]:
res = least_squares(rootfunc, [-15.5*AU, -48*AU, 0.0], bounds=([-100*AU,-100*AU,0.0], [100*AU,100*AU,4.0]))

70.0 [-8.84022166e+09 -3.88877844e+09 -2.90522059e+09]
70.0 [-8.84022166e+09 -3.88877848e+09 -2.90522059e+09]
70.0 [-8.84022166e+09 -3.88877844e+09 -2.90522070e+09]
70.0 [-8.84022163e+09 -3.88877844e+09 -2.90522059e+09]
70.0 [-4.39359224e+09 -7.22902493e+08 -3.20466846e+08]
70.0 [-4.39359224e+09 -7.22902480e+08 -3.20466846e+08]
70.0 [-4.39359224e+09 -7.22902493e+08 -3.20466913e+08]
70.0 [-4.39359217e+09 -7.22902493e+08 -3.20466845e+08]
70.0 [-2.17045990e+09 -3.93881778e+07  9.13129411e+06]
70.0 [-2.17045990e+09 -3.93881546e+07  9.13129410e+06]
70.0 [-2.17045990e+09 -3.93881778e+07  9.13123171e+06]
70.0 [-2.17045980e+09 -3.93881779e+07  9.13129439e+06]
70.0 [-1.05911229e+09 -3.05293111e+06  7.86918193e+06]
70.0 [-1.05911229e+09 -3.05290736e+06  7.86918192e+06]
70.0 [-1.05911229e+09 -3.05293108e+06  7.86911951e+06]
70.0 [-1.05911217e+09 -3.05293125e+06  7.86918229e+06]
70.0 [-5.04061138e+08 -1.51217825e+06  3.96955119e+06]
70.0 [-5.04061138e+08 -1.51215447e+06  3.96955118e+06]
70.0 [-5.0

In [11]:
res['message']

'`xtol` termination condition is satisfied.'

In [12]:
human_res = res['x'].copy()
human_res[0] /= AU
human_res[1] /= AU

print(human_res)


[ 10.86263103 -28.51480415   3.95211844]


In [13]:
res = solve_ivp(dydt, [0, 70*YEAR], [-200*AU, human_res[0]*AU, human_res[1]*AU, human_res[2], 0.0, 0.0], events=event, method='DOP853', rtol=1e-12, atol=1e-12)

In [14]:
res['t'] / YEAR

array([0.00000000e+00, 1.07710068e-08, 1.18481075e-07, 1.19558176e-06,
       1.19665886e-05, 1.19676657e-04, 1.19677734e-03, 1.19677842e-02,
       1.19677852e-01, 1.19677853e+00, 1.19677854e+01, 2.92870022e+01,
       4.33577613e+01, 5.74285203e+01, 7.00000000e+01])

In [15]:
(res['y'][0:3]).T / AU

array([[-200.        ,   10.86263103,  -28.51480415],
       [-199.99999999,   10.86263103,  -28.51480415],
       [-199.9999999 ,   10.86263103,  -28.51480415],
       [-199.999999  ,   10.86263103,  -28.51480415],
       [-199.99999002,   10.86263103,  -28.51480415],
       [-199.99990023,   10.86263103,  -28.51480415],
       [-199.99900225,   10.86263103,  -28.51480415],
       [-199.99002242,   10.86263103,  -28.51480414],
       [-199.9002177 ,   10.86263064,  -28.51480312],
       [-199.00152946,   10.8625919 ,  -28.51470143],
       [-189.94839547,   10.85853708,  -28.50405738],
       [-175.11675323,   10.83613161,  -28.44524221],
       [-162.78357547,   10.80043234,  -28.3515303 ],
       [-150.15529392,   10.74510044,  -28.20628205],
       [-138.58320844,   10.67495418,  -28.02214553]])

In [16]:
rel_r, rel_v = rel_rv_from_X(res['t'][-1], res['y'][:,-1].T)

In [17]:
rel_r, rel_v

(array([-2.07519531e-03, -5.48362732e-06,  1.90734863e-05]),
 array([4.5118141 , 1.59754453, 1.40185717]))

In [18]:
# first sol:
sol_line_1_raw = [
    '0', '0', '0.0', [-200*AU, human_res[0]*AU, human_res[1]*AU], [human_res[2], 0.0, 0.0], [0.0, 0.0, 0.0]
]

sol_line_2_raw = [
    '0', '0', res['t'][-1], res['y'][0:3,-1].T, res['y'][3:6,-1].T, [0.0, 0.0, 0.0]
]

planetx_id = float(ss.init_bodies.loc[planetX]['id'].to_numpy().squeeze())

sol_line_3_raw = [
    planetx_id, '1', res['t'][-1], res['y'][0:3,-1].T, res['y'][3:6,-1].T, rel_v
]

In [19]:
def flatten_list(l):
    out = []
    for v in l:
        if isinstance(v, (int, float, str)):
            out.append(float(v))
        elif isinstance(v, (list, np.ndarray)):
            for y in v:
                out.append(float(y))
    return out

In [20]:
sol_lines = list()
sol_lines.append(flatten_list(sol_line_1_raw))
sol_lines.append(flatten_list(sol_line_2_raw))
sol_lines.append(flatten_list(sol_line_3_raw))

In [21]:
sol_lines

[[0.0,
  0.0,
  0.0,
  -29919574138.200005,
  1625026472.2571545,
  -4265753983.5165567,
  3.9521184447318487,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0],
 [0.0,
  0.0,
  2209032000.0,
  -20731752896.68302,
  1596950414.5640807,
  -4192053304.179736,
  4.420677784943152,
  -0.03074058090072269,
  0.08069515030775429,
  0.0,
  0.0,
  0.0],
 [10.0,
  1.0,
  2209032000.0,
  -20731752896.68302,
  1596950414.5640807,
  -4192053304.179736,
  4.420677784943152,
  -0.03074058090072269,
  0.08069515030775429,
  4.511814100161608,
  1.5975445285723726,
  1.4018571667164517]]

In [22]:
import csv

with open('planetx_intercept.csv', 'w', newline='') as f:
    header = ['#body_id','flag','epoch','rx','ry','rz','vx','vy','vz','ux','uy','uz']
    writer = csv.writer(f)
    writer.writerow(header)
    for l in sol_lines:
        writer.writerow(l)

In [23]:
# calculate time to periapsis
def event2(t,y):
    r = y[0:3]; v = y[3:6]
    return np.dot(r,v)

event2.terminal = True
    
res = solve_ivp(dydt, [0, 200*YEAR], [-200*AU, human_res[0]*AU, human_res[1]*AU, human_res[2], 0.0, 0.0], events=event2, method='DOP853', rtol=1e-12, atol=1e-12)

In [24]:
res['t'] / YEAR

array([0.00000000e+00, 1.07710068e-08, 1.18481075e-07, 1.19558176e-06,
       1.19665886e-05, 1.19676657e-04, 1.19677734e-03, 1.19677842e-02,
       1.19677852e-01, 1.19677853e+00, 1.19677854e+01, 2.92870022e+01,
       4.33577613e+01, 5.74285203e+01, 6.87542688e+01, 8.00800173e+01,
       8.92481157e+01, 9.84162142e+01, 1.05876844e+02, 1.13337474e+02,
       1.19438799e+02, 1.25540123e+02, 1.31097744e+02, 1.36110929e+02,
       1.40636422e+02, 1.44726966e+02, 1.48429183e+02, 1.51784256e+02,
       1.54828628e+02, 1.57594610e+02, 1.60110908e+02, 1.62403081e+02,
       1.64493933e+02, 1.66403865e+02, 1.68151180e+02, 1.69752354e+02,
       1.71222284e+02, 1.72574519e+02, 1.73821481e+02, 1.74974700e+02,
       1.76045074e+02, 1.77043195e+02, 1.77979816e+02, 1.78866493e+02,
       1.79716120e+02, 1.80539037e+02, 1.81183590e+02, 1.81734172e+02,
       1.82225908e+02, 1.82717643e+02, 1.83179950e+02, 1.83593721e+02,
       1.84007493e+02, 1.84391365e+02, 1.84752035e+02, 1.85090096e+02,
      

In [25]:
res['y'][0:3,-1] / AU

array([ 7.1460173 ,  0.92855695, -2.43749598])